# What FlashAttention saves
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/demos/week4/FlashAttention_Memory_Demo_Colab.ipynb)

**Dataflow Architectures for AI, Week 4, Module 5: Tiling, layout and FlashAttention**

Standard attention computes the score matrix `S = Q K^T`, writes it to GPU memory, reads it back for the softmax, writes the probabilities `P`, and reads them again to multiply by `V`. `S` and `P` are both N by N, so their size grows with the square of the sequence length. FlashAttention is tiling applied to attention: it loads one block of `Q`, streams blocks of `K` and `V` past it, keeps each score tile on chip, and uses a running maximum and sum (online softmax) so the output stays correct. The N by N matrix is never written to memory.

In this notebook you measure the peak memory and time of attention as the sequence grows, for a hand-written version and for each backend of PyTorch's `scaled_dot_product_attention`, and you compare the results with the lecture's byte counts.

**What you will do**
1. Compute the size of the score matrix for each sequence length.
2. Run hand-written attention and record its peak memory and time.
3. Run `scaled_dot_product_attention` with the math, memory-efficient and FlashAttention backends.
4. Check that the backends agree on the result.
5. Compare the memory traffic with the lecture's numbers.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. The notebook takes about a minute.

## 1. Check the environment
PyTorch has three attention backends. **math** is the reference implementation and materializes the scores. **memory-efficient** is a tiled kernel that works on most GPUs. **FlashAttention** in PyTorch needs compute capability 8.0 or later, so on a T4 it is expected to be unavailable. The cell prints the GPU's compute capability.

In [ ]:
import time
import warnings
import torch
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

assert torch.cuda.is_available(), "This notebook needs a GPU: Runtime > Change runtime type > T4 GPU"
cc = torch.cuda.get_device_capability(0)
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0), f"| compute capability {cc[0]}.{cc[1]}")

B, H, D = 1, 8, 64                       # batch, heads, head dimension
dtype = torch.float16

## 2. The size of the score matrix
For one head, `S` has N x N entries. At FP16 that is `2 x N^2` bytes per head. The lecture example is 8K tokens: about 134 MB per head. Writing and reading both `S` and `P` once is four times that, about 537 MB of memory traffic per head.

`Q`, `K`, `V` and the output `O` are each `N x D` per head, which is small by comparison.

In [ ]:
print(f"{'N':>6} {'S per head':>11} {'S, all heads':>13} {'S and P traffic':>16} {'Q, K, V, O':>11}")
for N in (1024, 2048, 4096, 8192):
    s_head = 2 * N * N
    qkvo = 4 * B * H * N * D * 2
    print(f"{N:6d} {s_head / 1e6:8.1f} MB {s_head * H / 1e6:10.0f} MB {4 * s_head * H / 1e6:13.0f} MB {qkvo / 1e6:8.1f} MB")

**Reading the output**
- At 8192 tokens the score matrix is 134 MB per head, as on the slide. With 8 heads, `S` alone is about 1 GB.
- Doubling N multiplies `S` by four. `Q`, `K`, `V` and `O` only double.

## 3. Measure each implementation
The helper runs one attention call and reports the extra memory it allocated (the peak minus the inputs) and the time. The hand-written version follows the four steps of standard attention one by one.

In [ ]:
def naive_attention(q, k, v):
    s = (q @ k.transpose(-2, -1)) * (q.shape[-1] ** -0.5)   # N x N scores, written to memory
    p = torch.softmax(s, dim=-1)                              # N x N probabilities
    return p @ v


def sdpa(backend):
    def run(q, k, v):
        # When a backend cannot run on this GPU, PyTorch prints one warning per backend it
        # tried before raising an error. The error message below is enough, so hide them.
        with sdpa_kernel(backend), warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return F.scaled_dot_product_attention(q, k, v)
    return run


impls = {
    "hand-written": naive_attention,
    "SDPA math": sdpa(SDPBackend.MATH),
    "SDPA memory-efficient": sdpa(SDPBackend.EFFICIENT_ATTENTION),
    "SDPA FlashAttention": sdpa(SDPBackend.FLASH_ATTENTION),
}


def measure(fn, N, iters=10):
    q, k, v = (torch.randn(B, H, N, D, device="cuda", dtype=dtype) for _ in range(3))
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    base = torch.cuda.memory_allocated()
    out = fn(q, k, v)
    torch.cuda.synchronize()
    extra = torch.cuda.max_memory_allocated() - base
    t0 = time.perf_counter()
    for _ in range(iters):
        fn(q, k, v)
    torch.cuda.synchronize()
    return extra, (time.perf_counter() - t0) / iters * 1e3, (q, k, v, out)


results = {}
for name, fn in impls.items():
    print(name)
    for N in (1024, 2048, 4096, 8192):
        try:
            extra, ms, _ = measure(fn, N)
            results[(name, N)] = (extra, ms)
            print(f"  N = {N:5d}: extra memory {extra / 1e6:8.1f} MB, {ms:7.2f} ms")
        except RuntimeError as err:
            print(f"  N = {N:5d}: not available on this GPU ({str(err).splitlines()[0][:80]})")
            break
    torch.cuda.empty_cache()

**Reading the output**
- The hand-written version and the math backend allocate memory that grows four times when N doubles. At 8192 tokens they allocate more than 2 GB: the score and probability matrices, plus temporaries.
- The memory-efficient backend allocates about the size of the output, a few MB, at every length. The scores exist only as tiles inside the kernel.
- On a T4, the FlashAttention backend reports that no kernel is available: PyTorch's FlashAttention kernels support compute capability 8.0 and newer. On an A100, L4 or H100 it runs and behaves like the memory-efficient backend.
- The tiled kernel is also faster, because it does not write and read the N by N matrices.

## 4. Do they agree?
A faster kernel is only useful if it computes the same thing. The cell compares each available backend with the hand-written version at 2048 tokens.

In [ ]:
torch.manual_seed(0)
q, k, v = (torch.randn(B, H, 2048, D, device="cuda", dtype=dtype) for _ in range(3))
ref = naive_attention(q.float(), k.float(), v.float())   # FP32 reference
for name, fn in impls.items():
    try:
        out = fn(q, k, v).float()
        print(f"{name:22}: max difference from the FP32 reference {(out - ref).abs().max().item():.2e}")
    except RuntimeError:
        print(f"{name:22}: not available")

**Reading the output**
- All available implementations agree with the FP32 reference to between 1e-4 and 1e-3, within FP16 rounding. The online softmax is exact, not an approximation: the order of the additions changes, and FP16 rounding shows that.

## 5. Traffic compared with the lecture
The cell estimates the memory traffic per head for standard attention (write and read `S` and `P`) and for the tiled kernel (read `Q`, `K`, `V` once and write `O`), and converts it into a time at the T4's 320 GB/s.

In [ ]:
print(f"{'N':>6} {'standard, per head':>19} {'tiled, per head':>16} {'ratio':>6} {'standard time, 8 heads':>23}")
for N in (1024, 2048, 4096, 8192):
    standard = 4 * 2 * N * N + 4 * N * D * 2
    tiled = 4 * N * D * 2
    print(f"{N:6d} {standard / 1e6:16.1f} MB {tiled / 1e6:13.2f} MB {standard / tiled:6.0f} {standard * H / 320e9 * 1e3:20.2f} ms")

**Reading the output**
- At 8192 tokens standard attention moves about 537 MB per head, and the tiled kernel's minimum is about 4 MB, as on the slide.
- The tiled number is a lower bound. The kernel rereads `K` and `V` once for every block of `Q`. On a GPU the L2 cache serves most of those rereads. On Spyre, which has no L2 cache, the compiler has to plan that sharing, which is the lecture's next slide.
- FlashAttention does not make the KV cache smaller. It removes the N by N intermediate. Module 6 covers the KV cache itself.

## Same notebook on other GPUs
The same notebook was run on four NVIDIA GPUs: the T4 from Colab's free tier, the L4, the A100 and the H100. The lecture marks values like these as MEASURED, observed on hardware, and SNAPSHOT, valid for one software release. Read the table for the pattern across GPUs, not for exact numbers.

<!-- gpu-table:start -->
| GPU | hand-written | memory-efficient | FlashAttention |
|---|---|---|---|
| T4 (cc 7.5) | 2,164 MB, 35.47 ms | 8 MB, 9.41 ms | not available |
| L4 (cc 8.9) | 2,164 MB, 27.46 ms | 8 MB, 3.60 ms | 9 MB, 2.25 ms |
| A100-80GB (cc 8.0) | 2,164 MB, 7.22 ms | 8 MB, 1.83 ms | 9 MB, 0.80 ms |
| H100 (cc 9.0) | 2,189 MB, 3.62 ms | 8 MB, 1.08 ms | 9 MB, 0.46 ms |

Each cell is the extra memory and the time for one call at N = 8192, 8 heads.

*MEASURED on cloud GPUs on 2026-11-18, PyTorch 2.15.0. SNAPSHOT: these values change with library versions and with the host CPU, and your Colab T4 can differ from the T4 row.*
<!-- gpu-table:end -->

**What the comparison shows**
- The FlashAttention backend runs on the L4, A100 and H100, which have compute capability 8.0 or later. On the T4 it is not available.
- Memory behaves the same on every GPU. The hand-written version needs about 2 GB at 8192 tokens everywhere, and the tiled kernels allocate only about the size of the output.
- Where FlashAttention runs, it is the fastest backend, and its lead over the memory-efficient kernel is largest on the newest GPU.

## Try this
1. Run section 3 with N = 16384 for the memory-efficient backend only. How much memory would the hand-written version need?
2. Change `D` to 128 and run section 5. Which column changes, and by how much?
3. Add a causal mask: pass `is_causal=True` to `scaled_dot_product_attention`. Does the memory-efficient backend get faster?

## Summary
- Standard attention writes and reads two N by N matrices per head, so its memory and traffic grow with the square of the sequence length.
- FlashAttention-style kernels keep score tiles on chip and use an online softmax. They allocate only the output and move far fewer bytes.
- The result is the same up to rounding.
- PyTorch's FlashAttention backend needs a newer GPU than the T4. The memory-efficient backend gives the same memory behavior on the T4.

Lecture: Week 4, Module 5, slides 43 and 44 (FlashAttention, FlashAttention on Spyre).